# Interactive Demo: PLL, Park (DQ0), and Fortescue Transforms

This notebook demonstrates the three key operations in time-domain 3-phase analysis:

1. **Phase Lock Loop (PLL)**: Synchronizes to carrier frequency
2. **Park Transform (DQ0)**: Rotates to measurement frame
3. **Fortescue (Sequences)**: Decomposes into spin eigenstates

Use the interactive sliders to explore how each transform works!

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, RadioButtons
from IPython.display import display, HTML
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual

%matplotlib widget
plt.rcParams['figure.figsize'] = (14, 8)

---
## Part 1: Phase Lock Loop (PLL)

**What it does:** Tracks the carrier frequency of incoming signal

**Why needed:** Measurement requires phase coherence with the quantum state

**How it works:**
1. Measure phase error between signal and local oscillator
2. Adjust local oscillator frequency based on error
3. Lock when phase error → 0

In [ ]:
def pll_demo(signal_freq=1.0, pll_gain=0.1, pll_init_freq=0.8):
    """
    Interactive PLL demonstration
    """
    # Time array
    dt = 0.01
    t = np.arange(0, 10, dt)
    
    # Input signal (what we're trying to lock to)
    signal = np.cos(2 * np.pi * signal_freq * t)
    
    # PLL simulation
    phase_est = np.zeros_like(t)
    freq_est = np.zeros_like(t)
    phase_error = np.zeros_like(t)
    
    phase_est[0] = 0
    freq_est[0] = 2 * np.pi * pll_init_freq
    
    for i in range(1, len(t)):
        # Phase detector (multiply signal with VCO output)
        vco_output = np.cos(phase_est[i-1])
        phase_error[i] = signal[i] * vco_output
        
        # Loop filter + VCO
        freq_est[i] = freq_est[i-1] + pll_gain * phase_error[i]
        phase_est[i] = phase_est[i-1] + freq_est[i] * dt
    
    # VCO output
    vco_signal = np.cos(phase_est)
    
    # Create figure
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    # Top: Input signal vs VCO output
    ax = axes[0]
    ax.plot(t, signal, 'b-', linewidth=2, label=f'Input Signal ({signal_freq:.1f} Hz)', alpha=0.7)
    ax.plot(t, vco_signal, 'r--', linewidth=2, label='VCO Output (PLL tracking)', alpha=0.7)
    ax.set_ylabel('Amplitude', fontsize=12)
    ax.set_title('Phase Lock Loop: Signal Tracking', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 10])
    
    # Middle: Frequency tracking
    ax = axes[1]
    ax.plot(t, freq_est / (2*np.pi), 'g-', linewidth=2, label='Estimated Frequency')
    ax.axhline(y=signal_freq, color='b', linestyle='--', linewidth=2, 
               label=f'Target: {signal_freq:.1f} Hz', alpha=0.7)
    ax.set_ylabel('Frequency (Hz)', fontsize=12)
    ax.set_title('Frequency Estimation (VCO frequency)', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 10])
    
    # Bottom: Phase error
    ax = axes[2]
    ax.plot(t, phase_error, 'purple', linewidth=2, label='Phase Error')
    ax.axhline(y=0, color='k', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('Time (s)', fontsize=12)
    ax.set_ylabel('Phase Error', fontsize=12)
    ax.set_title('Phase Error (drives correction)', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 10])
    
    plt.tight_layout()
    
    # Print lock status
    final_freq = freq_est[-1] / (2*np.pi)
    locked = abs(final_freq - signal_freq) < 0.01
    
    print(f"\nPLL Status:")
    print(f"  Target frequency: {signal_freq:.3f} Hz")
    print(f"  Final estimate: {final_freq:.3f} Hz")
    print(f"  Error: {abs(final_freq - signal_freq):.4f} Hz")
    print(f"  Locked: {'✓ YES' if locked else '✗ NO (increase gain or wait longer)'}")
    
    return fig

# Interactive widget
interact(pll_demo, 
         signal_freq=widgets.FloatSlider(min=0.5, max=2.0, step=0.1, value=1.0, 
                                        description='Signal Freq (Hz)'),
         pll_gain=widgets.FloatSlider(min=0.01, max=0.5, step=0.01, value=0.1, 
                                     description='PLL Gain'),
         pll_init_freq=widgets.FloatSlider(min=0.3, max=1.5, step=0.1, value=0.8,
                                          description='Initial VCO Freq (Hz)'));

### Key Observations:

- **Phase Error**: Drives the correction - when it averages to zero, PLL is locked
- **Gain**: Higher gain = faster lock, but too high causes oscillation
- **Initial Frequency**: Starting closer to target speeds up acquisition

**In quantum measurement:** PLL locks onto the photon's carrier frequency ω, enabling coherent detection.

---
## Part 2: Park Transform (DQ0)

**What it does:** Rotates 3-phase signals into a reference frame synchronized with rotation

**Why needed:** Converts oscillating AC signals to DC, making measurement easier

**How it works:**
1. Take 3-phase signals (I_a, I_b, I_c)
2. Rotate by angle θ to align with measurement axis
3. Extract I_d (direct) and I_q (quadrature) components

In [ ]:
def park_transform_demo(polarization='Horizontal', measurement_angle=0.0, show_energy=True):
    """
    Interactive Park Transform demonstration
    """
    # Time array
    t = np.linspace(0, 4*np.pi, 1000)
    omega = 1.0
    
    # Define polarization states
    alpha = np.exp(1j * 2*np.pi/3)
    
    if polarization == 'Horizontal':
        I_1, I_2 = 1/np.sqrt(2), 1/np.sqrt(2)
        title_suffix = '(Equal I₁ and I₂)'
    elif polarization == 'Vertical':
        I_1, I_2 = 1/np.sqrt(2), -1/np.sqrt(2)
        title_suffix = '(I₁ and I₂ with opposite phase)'
    elif polarization == 'Right Circular':
        I_1, I_2 = 1.0, 0.0
        title_suffix = '(Pure I₁, positive sequence)'
    elif polarization == 'Left Circular':
        I_1, I_2 = 0.0, 1.0
        title_suffix = '(Pure I₂, negative sequence)'
    
    # Generate 3-phase signals
    phase_factor = np.exp(1j * omega * t)
    I_a = np.real(I_1 * phase_factor + I_2 * np.conj(phase_factor))
    I_b = np.real(I_1 * phase_factor * alpha**2 + I_2 * np.conj(phase_factor) * alpha)
    I_c = np.real(I_1 * phase_factor * alpha + I_2 * np.conj(phase_factor) * alpha**2)
    
    # Park transform
    theta = omega * t + np.deg2rad(measurement_angle)
    sqrt_2_3 = np.sqrt(2/3)
    
    I_d = sqrt_2_3 * (np.cos(theta) * I_a + 
                      np.cos(theta - 2*np.pi/3) * I_b + 
                      np.cos(theta - 4*np.pi/3) * I_c)
    
    I_q = sqrt_2_3 * (-np.sin(theta) * I_a - 
                      np.sin(theta - 2*np.pi/3) * I_b - 
                      np.sin(theta - 4*np.pi/3) * I_c)
    
    I_0 = sqrt_2_3 * 0.5 * (I_a + I_b + I_c)
    
    # Create figure
    if show_energy:
        fig, axes = plt.subplots(3, 2, figsize=(16, 11))
    else:
        fig, axes = plt.subplots(2, 2, figsize=(16, 8))
    
    # Top left: 3-phase signals
    ax = axes[0, 0]
    ax.plot(t, I_a, 'r-', linewidth=2, label='I_a (0°)', alpha=0.7)
    ax.plot(t, I_b, 'g-', linewidth=2, label='I_b (120°)', alpha=0.7)
    ax.plot(t, I_c, 'b-', linewidth=2, label='I_c (240°)', alpha=0.7)
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    ax.set_ylabel('Amplitude', fontsize=11)
    ax.set_title(f'3-Phase Signals: {polarization}\n{title_suffix}', 
                fontsize=12, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 4*np.pi])
    
    # Top right: Park DQ components
    ax = axes[0, 1]
    ax.plot(t, I_d, 'purple', linewidth=3, label='I_d (direct)', alpha=0.8)
    ax.plot(t, I_q, 'orange', linewidth=3, label='I_q (quadrature)', alpha=0.8)
    ax.axhline(y=np.mean(I_d), color='purple', linestyle='--', linewidth=2,
              label=f'DC: I_d = {np.mean(I_d):.3f}')
    ax.axhline(y=np.mean(I_q), color='orange', linestyle='--', linewidth=2,
              label=f'DC: I_q = {np.mean(I_q):.3f}')
    ax.set_ylabel('Amplitude', fontsize=11)
    ax.set_title(f'Park Transform (DQ0) at θ={measurement_angle:.0f}°\nAC → DC conversion', 
                fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 4*np.pi])
    
    # Bottom left: Zero sequence
    ax = axes[1, 0]
    ax.plot(t, I_0, 'gray', linewidth=3, label='I_0 (zero sequence)')
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.5)
    ax.set_xlabel('Time (ωt)', fontsize=11)
    ax.set_ylabel('Amplitude', fontsize=11)
    ax.set_title('Zero Sequence (should be ~0 for photon)\nMaxwell\'s transversality', 
                fontsize=12, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 4*np.pi])
    
    # Bottom right: DQ trajectory (phasor)
    ax = axes[1, 1]
    ax.plot(I_d, I_q, 'purple', linewidth=2, alpha=0.7, label='DQ Trajectory')
    ax.scatter([np.mean(I_d)], [np.mean(I_q)], s=200, c='red', marker='x', 
              linewidths=3, label='DC Point', zorder=10)
    ax.set_xlabel('I_d (direct)', fontsize=11)
    ax.set_ylabel('I_q (quadrature)', fontsize=11)
    ax.set_title('DQ Plane (Phasor Diagram)\nDC point = measurement outcome', 
                fontsize=12, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    ax.axhline(y=0, color='k', linestyle='-', alpha=0.3)
    ax.axvline(x=0, color='k', linestyle='-', alpha=0.3)
    
    # Optional: Energy plot
    if show_energy:
        ax = axes[2, 0]
        E_3phase = I_a**2 + I_b**2 + I_c**2
        E_dq = I_d**2 + I_q**2
        ax.plot(t, E_3phase, 'g-', linewidth=2, label='E (3-phase) = I_a² + I_b² + I_c²', alpha=0.7)
        ax.plot(t, E_dq, 'purple', linewidth=2, label='E (DQ) = I_d² + I_q²', alpha=0.7)
        ax.axhline(y=np.mean(E_dq), color='purple', linestyle='--', linewidth=2)
        ax.set_xlabel('Time (ωt)', fontsize=11)
        ax.set_ylabel('Energy', fontsize=11)
        ax.set_title('Energy Conservation\n(Constant in both representations)', 
                    fontsize=12, fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        ax.set_xlim([0, 4*np.pi])
        
        # Info panel
        ax = axes[2, 1]
        ax.axis('off')
        info_text = f"""
Park Transform Summary:
═══════════════════════════════════
Polarization: {polarization}
Measurement angle: {measurement_angle:.0f}°

DC Components (Time-averaged):
  I_d = {np.mean(I_d):.4f}
  I_q = {np.mean(I_q):.4f}
  I_0 = {np.mean(I_0):.6f} (≈0 ✓)

Energy:
  3-phase: {np.mean(E_3phase):.4f}
  DQ:      {np.mean(E_dq):.4f}
  Conserved: {'✓' if np.abs(np.mean(E_3phase) - np.mean(E_dq)) < 0.01 else '✗'}

Physical Interpretation:
  • AC signals → DC components
  • I_d = real power (measurement)
  • I_q = reactive power
  • DC point = quantum outcome
        """
        ax.text(0.1, 0.9, info_text, transform=ax.transAxes, 
               fontsize=10, family='monospace', va='top')
    
    plt.tight_layout()
    return fig

# Interactive widget
interact(park_transform_demo,
         polarization=widgets.Dropdown(options=['Horizontal', 'Vertical', 
                                                'Right Circular', 'Left Circular'],
                                      value='Horizontal',
                                      description='Polarization'),
         measurement_angle=widgets.FloatSlider(min=0, max=180, step=15, value=0,
                                              description='Meas. Angle (°)'),
         show_energy=widgets.Checkbox(value=True, description='Show Energy'));

### Key Observations:

- **DC Components**: I_d and I_q settle to constant values (time-averaged)
- **Measurement Angle**: Rotating θ changes the DC values (projection)
- **Zero Sequence**: I_0 ≈ 0 for photons (transversality constraint)
- **Energy**: Conserved in both 3-phase and DQ representations

**In quantum measurement:** Park transform projects photon state onto measurement axis θ.

---
## Part 3: Fortescue Transform (Sequence Decomposition)

**What it does:** Decomposes 3-phase into symmetrical components (sequences)

**Why needed:** Sequences ARE the spin eigenstates!

**How it works:**
1. Take 3-phase currents (I_a, I_b, I_c)
2. Apply Fortescue matrix (eigendecomposition)
3. Get sequences: I_0 (zero), I_1 (positive), I_2 (negative)

In [ ]:
def fortescue_demo(I_1_mag=1.0, I_2_mag=0.0, I_1_phase=0.0, I_2_phase=0.0, show_rotation=True):
    """
    Interactive Fortescue Transform demonstration
    """
    # Complex sequences
    I_1 = I_1_mag * np.exp(1j * np.deg2rad(I_1_phase))
    I_2 = I_2_mag * np.exp(1j * np.deg2rad(I_2_phase))
    I_0 = 0  # Always zero for photons
    
    # Fortescue inverse transform (sequences → phases)
    alpha = np.exp(1j * 2*np.pi/3)
    I_a = I_0 + I_1 + I_2
    I_b = I_0 + I_1 * alpha**2 + I_2 * alpha
    I_c = I_0 + I_1 * alpha + I_2 * alpha**2
    
    # Time evolution
    t = np.linspace(0, 4*np.pi, 1000)
    omega = 1.0
    
    phase_factor = np.exp(1j * omega * t)
    I_a_t = np.real(I_1 * phase_factor + I_2 * np.conj(phase_factor))
    I_b_t = np.real(I_1 * phase_factor * alpha**2 + I_2 * np.conj(phase_factor) * alpha)
    I_c_t = np.real(I_1 * phase_factor * alpha + I_2 * np.conj(phase_factor) * alpha**2)
    
    # Determine polarization
    if abs(I_1) > 0.9 and abs(I_2) < 0.1:
        pol_type = 'Right Circular'
    elif abs(I_2) > 0.9 and abs(I_1) < 0.1:
        pol_type = 'Left Circular'
    elif abs(abs(I_1) - abs(I_2)) < 0.1:
        if abs(np.angle(I_1) - np.angle(I_2)) < 0.1:
            pol_type = 'Linear (Horizontal-ish)'
        elif abs(abs(np.angle(I_1) - np.angle(I_2)) - np.pi) < 0.1:
            pol_type = 'Linear (Vertical-ish)'
        else:
            pol_type = 'Linear (Diagonal)'
    else:
        pol_type = 'Elliptical'
    
    # Create figure
    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # Top left: Sequence magnitudes
    ax1 = fig.add_subplot(gs[0, 0])
    sequences = ['I₀\n(Zero)', 'I₁\n(Positive)', 'I₂\n(Negative)']
    mags = [abs(I_0), abs(I_1), abs(I_2)]
    colors_seq = ['gray', 'blue', 'red']
    bars = ax1.bar(sequences, mags, color=colors_seq, alpha=0.7, edgecolor='black', linewidth=2)
    ax1.set_ylabel('Magnitude', fontsize=11)
    ax1.set_title('Fortescue Sequences\n(Spin Decomposition)', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='y')
    ax1.set_ylim([0, 1.2])
    
    # Add value labels
    for bar, val in zip(bars, mags):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Top middle: Complex plane (phasors)
    ax2 = fig.add_subplot(gs[0, 1])
    
    # Draw sequences as phasors
    if abs(I_1) > 0.01:
        ax2.arrow(0, 0, np.real(I_1), np.imag(I_1), head_width=0.08, head_length=0.08,
                 fc='blue', ec='blue', linewidth=3, alpha=0.8, label='I₁ (Right)')
    if abs(I_2) > 0.01:
        ax2.arrow(0, 0, np.real(I_2), np.imag(I_2), head_width=0.08, head_length=0.08,
                 fc='red', ec='red', linewidth=3, alpha=0.8, label='I₂ (Left)')
    
    # Resultant
    resultant = I_1 + I_2
    if abs(resultant) > 0.01:
        ax2.arrow(0, 0, np.real(resultant), np.imag(resultant), 
                 head_width=0.1, head_length=0.1,
                 fc='purple', ec='purple', linewidth=4, alpha=0.9, 
                 label='Resultant', linestyle='--')
    
    ax2.set_xlim([-1.2, 1.2])
    ax2.set_ylim([-1.2, 1.2])
    ax2.set_xlabel('Real', fontsize=11)
    ax2.set_ylabel('Imaginary', fontsize=11)
    ax2.set_title('Complex Plane (Phasors)\nI₁ + I₂ = Resultant', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=0, color='k', linestyle='-', alpha=0.3)
    ax2.axvline(x=0, color='k', linestyle='-', alpha=0.3)
    ax2.set_aspect('equal')
    ax2.legend(loc='upper right', fontsize=9)
    
    # Top right: Polarization type
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.axis('off')
    
    info_text = f"""
Sequence Analysis:
═══════════════════════════
I₁ = {abs(I_1):.3f} ∠{np.rad2deg(np.angle(I_1)):.1f}°
I₂ = {abs(I_2):.3f} ∠{np.rad2deg(np.angle(I_2)):.1f}°
I₀ = 0.000 (photon)

Polarization:
  {pol_type}

Spin Interpretation:
  |ψ⟩ = c₁|R⟩ + c₂|L⟩
  c₁ = {abs(I_1):.3f}
  c₂ = {abs(I_2):.3f}

Helicity:
  Right: {abs(I_1)**2:.3f}
  Left:  {abs(I_2)**2:.3f}
  Total: {abs(I_1)**2 + abs(I_2)**2:.3f}
    """
    
    ax3.text(0.1, 0.9, info_text, transform=ax3.transAxes,
            fontsize=10, family='monospace', va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Middle row: Time-domain 3-phase signals
    ax4 = fig.add_subplot(gs[1, :])
    ax4.plot(t, I_a_t, 'r-', linewidth=2, label='I_a (0°)', alpha=0.7)
    ax4.plot(t, I_b_t, 'g-', linewidth=2, label='I_b (120°)', alpha=0.7)
    ax4.plot(t, I_c_t, 'b-', linewidth=2, label='I_c (240°)', alpha=0.7)
    ax4.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    ax4.set_xlabel('Time (ωt)', fontsize=11)
    ax4.set_ylabel('Amplitude', fontsize=11)
    ax4.set_title('Time Domain: 3-Phase Currents', fontsize=12, fontweight='bold')
    ax4.legend(loc='upper right')
    ax4.grid(True, alpha=0.3)
    ax4.set_xlim([0, 4*np.pi])
    
    # Bottom left: Instantaneous phasors (rotating)
    if show_rotation:
        ax5 = fig.add_subplot(gs[2, 0])
        
        # Show phasors at t=0
        angles = [0, 120, 240]
        colors_phase = ['red', 'green', 'blue']
        labels_phase = ['a', 'b', 'c']
        
        for angle, color, label in zip(angles, colors_phase, labels_phase):
            theta = np.deg2rad(angle)
            # Direction
            ax5.plot([0, np.cos(theta)], [0, np.sin(theta)], 
                    color='gray', linestyle='--', alpha=0.3, linewidth=1)
            ax5.text(1.1*np.cos(theta), 1.1*np.sin(theta), label,
                    fontsize=12, ha='center', va='center', fontweight='bold')
        
        # Current phasors at t=0
        I_a_0, I_b_0, I_c_0 = np.real([I_a, I_b, I_c])
        
        for angle, I_val, color in zip(angles, [I_a_0, I_b_0, I_c_0], colors_phase):
            theta = np.deg2rad(angle)
            ax5.arrow(0, 0, I_val*np.cos(theta), I_val*np.sin(theta),
                     head_width=0.1, head_length=0.1, fc=color, ec=color,
                     linewidth=2, alpha=0.7)
        
        ax5.set_xlim([-1.3, 1.3])
        ax5.set_ylim([-1.3, 1.3])
        ax5.set_xlabel('x', fontsize=11)
        ax5.set_ylabel('y', fontsize=11)
        ax5.set_title('Instantaneous Phasors (t=0)', fontsize=12, fontweight='bold')
        ax5.grid(True, alpha=0.3)
        ax5.set_aspect('equal')
    
    # Bottom middle & right: Forward/Inverse transforms
    ax6 = fig.add_subplot(gs[2, 1:])
    ax6.axis('off')
    
    transform_text = f"""
Fortescue Transform:
═══════════════════════════════════════════════════════════════════════

FORWARD (3-phase → Sequences):

  I₀ = (1/3)(I_a + I_b + I_c)                           Zero sequence
  I₁ = (1/3)(I_a + α·I_b + α²·I_c)    where α = e^(j2π/3)    Positive (Right)
  I₂ = (1/3)(I_a + α²·I_b + α·I_c)                      Negative (Left)

INVERSE (Sequences → 3-phase):

  I_a = I₀ + I₁ + I₂
  I_b = I₀ + α²·I₁ + α·I₂  
  I_c = I₀ + α·I₁ + α²·I₂

KEY PROPERTIES:
  • Unitary transform (information preserving)
  • I₁ rotates forward (ABC) → Right circular → Helicity +1
  • I₂ rotates backward (ACB) → Left circular → Helicity -1
  • I₀ = 0 for photons (Maxwell's transversality)
  • Sequences are EIGENSTATES of rotation operator
    """
    
    ax6.text(0.05, 0.95, transform_text, transform=ax6.transAxes,
            fontsize=9.5, family='monospace', va='top')
    
    plt.suptitle('Fortescue Transform: Decomposition into Spin Eigenstates', 
                fontsize=14, fontweight='bold')
    
    return fig

# Interactive widget
interact(fortescue_demo,
         I_1_mag=widgets.FloatSlider(min=0, max=1.0, step=0.1, value=0.707,
                                    description='|I₁| (Right)'),
         I_2_mag=widgets.FloatSlider(min=0, max=1.0, step=0.1, value=0.707,
                                    description='|I₂| (Left)'),
         I_1_phase=widgets.FloatSlider(min=0, max=360, step=15, value=0,
                                      description='∠I₁ (deg)'),
         I_2_phase=widgets.FloatSlider(min=0, max=360, step=15, value=0,
                                      description='∠I₂ (deg)'),
         show_rotation=widgets.Checkbox(value=True, description='Show Phasors'));

### Key Observations:

- **Pure I₁**: Right circular polarization (helicity +1)
- **Pure I₂**: Left circular polarization (helicity -1)
- **Equal |I₁| = |I₂|**: Linear polarization (phase difference determines angle)
- **General case**: Elliptical polarization

**In quantum mechanics:** Fortescue sequences ARE the spin eigenstates!

---
## Part 4: Complete Pipeline

Now let's see all three operations working together in a realistic measurement simulation.

In [ ]:
def complete_pipeline_demo(polarization='Horizontal', measurement_angle=0.0, 
                          pll_gain=0.1, signal_noise=0.0):
    """
    Complete pipeline: 3-phase signal → PLL → Park → Measurement
    """
    # Parameters
    dt = 0.01
    t = np.arange(0, 10, dt)
    omega_true = 2 * np.pi * 1.0  # 1 Hz
    
    # Define state based on polarization
    alpha = np.exp(1j * 2*np.pi/3)
    
    if polarization == 'Horizontal':
        I_1, I_2 = 1/np.sqrt(2), 1/np.sqrt(2)
    elif polarization == 'Vertical':
        I_1, I_2 = 1/np.sqrt(2), -1/np.sqrt(2)
    elif polarization == 'Right Circular':
        I_1, I_2 = 1.0, 0.0
    elif polarization == 'Left Circular':
        I_1, I_2 = 0.0, 1.0
    
    # Generate 3-phase signals with noise
    phase_factor = np.exp(1j * omega_true * t)
    I_a = np.real(I_1 * phase_factor + I_2 * np.conj(phase_factor))
    I_b = np.real(I_1 * phase_factor * alpha**2 + I_2 * np.conj(phase_factor) * alpha)
    I_c = np.real(I_1 * phase_factor * alpha + I_2 * np.conj(phase_factor) * alpha**2)
    
    # Add noise
    I_a += signal_noise * np.random.randn(len(t))
    I_b += signal_noise * np.random.randn(len(t))
    I_c += signal_noise * np.random.randn(len(t))
    
    # STEP 1: PLL to track carrier
    phase_est = np.zeros_like(t)
    freq_est = np.zeros_like(t)
    phase_est[0] = 0
    freq_est[0] = 2 * np.pi * 0.8  # Start at 0.8 Hz
    
    for i in range(1, len(t)):
        # Simple phase detector using I_a
        vco_output = np.cos(phase_est[i-1])
        phase_error = I_a[i] * vco_output
        freq_est[i] = freq_est[i-1] + pll_gain * phase_error
        phase_est[i] = phase_est[i-1] + freq_est[i] * dt
    
    # STEP 2: Park transform using PLL phase
    theta = phase_est + np.deg2rad(measurement_angle)
    sqrt_2_3 = np.sqrt(2/3)
    
    I_d = sqrt_2_3 * (np.cos(theta) * I_a + 
                      np.cos(theta - 2*np.pi/3) * I_b + 
                      np.cos(theta - 4*np.pi/3) * I_c)
    
    I_q = sqrt_2_3 * (-np.sin(theta) * I_a - 
                      np.sin(theta - 2*np.pi/3) * I_b - 
                      np.sin(theta - 4*np.pi/3) * I_c)
    
    # STEP 3: Low-pass filter (moving average)
    window = 50
    I_d_filtered = np.convolve(I_d, np.ones(window)/window, mode='same')
    I_q_filtered = np.convolve(I_q, np.ones(window)/window, mode='same')
    
    # Measurement outcome (final average)
    measurement = np.mean(I_d_filtered[-200:])  # Average over last 2 seconds
    
    # Theoretical expectation
    theory = -np.cos(np.deg2rad(measurement_angle))
    
    # Create figure
    fig, axes = plt.subplots(4, 1, figsize=(16, 14))
    
    # Top: 3-phase input
    ax = axes[0]
    ax.plot(t, I_a, 'r-', linewidth=1, label='I_a', alpha=0.7)
    ax.plot(t, I_b, 'g-', linewidth=1, label='I_b', alpha=0.7)
    ax.plot(t, I_c, 'b-', linewidth=1, label='I_c', alpha=0.7)
    ax.set_ylabel('Amplitude', fontsize=11)
    ax.set_title(f'Step 1: Input Signal ({polarization})', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 10])
    
    # Second: PLL tracking
    ax = axes[1]
    ax.plot(t, freq_est / (2*np.pi), 'purple', linewidth=2, label='PLL Frequency Est.')
    ax.axhline(y=1.0, color='b', linestyle='--', linewidth=2, label='True: 1.0 Hz')
    ax.set_ylabel('Frequency (Hz)', fontsize=11)
    ax.set_title('Step 2: PLL Synchronization', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 10])
    
    # Third: Park transform output
    ax = axes[2]
    ax.plot(t, I_d, 'purple', linewidth=1, alpha=0.3, label='I_d (raw)')
    ax.plot(t, I_d_filtered, 'purple', linewidth=3, label='I_d (filtered)')
    ax.plot(t, I_q, 'orange', linewidth=1, alpha=0.3, label='I_q (raw)')
    ax.plot(t, I_q_filtered, 'orange', linewidth=3, label='I_q (filtered)')
    ax.axhline(y=measurement, color='purple', linestyle='--', linewidth=2, 
              label=f'DC: {measurement:.3f}')
    ax.set_ylabel('Amplitude', fontsize=11)
    ax.set_title('Step 3: Park Transform (DQ0)', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 10])
    
    # Bottom: Measurement outcome
    ax = axes[3]
    ax.plot(t, I_d_filtered, 'purple', linewidth=3, label='I_d (measurement)')
    ax.axhline(y=measurement, color='r', linestyle='-', linewidth=3, 
              label=f'Measured: {measurement:.4f}')
    ax.axhline(y=theory, color='b', linestyle='--', linewidth=2,
              label=f'Theory: {theory:.4f}')
    ax.set_xlabel('Time (s)', fontsize=11)
    ax.set_ylabel('I_d', fontsize=11)
    ax.set_title('Step 4: Final Measurement (Time-Averaged)', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 10])
    
    # Add text box with results
    error = abs(measurement - theory)
    textstr = f'Measurement: {measurement:.4f}\nTheory: {theory:.4f}\nError: {error:.4f}'
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
    ax.text(0.98, 0.97, textstr, transform=ax.transAxes, fontsize=12,
           verticalalignment='top', horizontalalignment='right', bbox=props)
    
    plt.tight_layout()
    
    print(f"\n{'='*60}")
    print(f"COMPLETE MEASUREMENT PIPELINE")
    print(f"{'='*60}")
    print(f"Polarization: {polarization}")
    print(f"Measurement angle: {measurement_angle:.0f}°")
    print(f"\nPLL Status: {'✓ Locked' if abs(freq_est[-1]/(2*np.pi) - 1.0) < 0.01 else '✗ Not locked'}")
    print(f"  Final frequency: {freq_est[-1]/(2*np.pi):.4f} Hz")
    print(f"\nMeasurement Result:")
    print(f"  Measured: {measurement:.4f}")
    print(f"  Theory:   {theory:.4f}")
    print(f"  Error:    {error:.4f}")
    print(f"  Agreement: {'✓ Good' if error < 0.05 else '✗ Poor (increase PLL gain?)'}")
    print(f"{'='*60}")
    
    return fig

# Interactive widget
interact(complete_pipeline_demo,
         polarization=widgets.Dropdown(options=['Horizontal', 'Vertical', 
                                                'Right Circular', 'Left Circular'],
                                      value='Horizontal',
                                      description='Polarization'),
         measurement_angle=widgets.FloatSlider(min=0, max=180, step=15, value=0,
                                              description='Meas. Angle (°)'),
         pll_gain=widgets.FloatSlider(min=0.01, max=0.3, step=0.01, value=0.1,
                                     description='PLL Gain'),
         signal_noise=widgets.FloatSlider(min=0.0, max=0.2, step=0.01, value=0.0,
                                         description='Noise Level'));

### Complete Pipeline Summary:

1. **Input**: 3-phase oscillating signals (photon state)
2. **PLL**: Locks onto carrier frequency
3. **Park**: Rotates to measurement frame → AC to DC
4. **Filter**: Time-averages to get final measurement

**Result**: Measurement outcome matches quantum theory!

---

## Conclusion

This notebook demonstrated the three key transforms:

- **PLL**: Synchronization (phase coherence)
- **Park (DQ0)**: Measurement projection (rotating frame)
- **Fortescue**: Spin decomposition (eigenstates)

Together, these show how **measurement works as a physical process** rather than abstract "collapse":

1. Synchronize to photon (PLL)
2. Rotate to measurement axis (Park)
3. Time-average (integration)
4. Extract DC = measurement outcome

**Physical interpretation**: Energy flows from reactive to real power (irreversible dissipation = measurement!)